In [9]:
import os
import json
import requests
import warnings
from PIL import Image, ExifTags, UnidentifiedImageError, ImageFile
import time

In [ ]:
# CONSTS
HEADERS = {
    "User-Agent": (
        "ImageRecoProjectBot/1.0 "
        "(https://exemple-projet-univ.fr; contact: prenom.nom@etu-votre-ecole.fr) "
        "requests/2.31.0"
    ),
    "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
    "Accept-Language": "fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7",
}
BASE_DIR = "."
IMAGES_DIR = os.path.join(BASE_DIR, "images")
DATA_DIR = os.path.join(BASE_DIR, "data")
METADATA_PATH = os.path.join(DATA_DIR, "images_metadata.json")

# Création des dossiers data et images
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
url_api = "https://commons.wikimedia.org/w/api.php?action=query&generator=search&gsrsearch=cat&gsrnamespace=6&gsrlimit=50&prop=imageinfo&iiprop=url|size|mime|extmetadata&format=json"

def search_commons_images(query, headers, limit=50):
    """
    Cherche des images sur Wikimedia Commons et renvoie une liste
    de dicts avec url + licence + auteur, etc.
    """
    params = {
        "action": "query",
        "generator": "search",
        "gsrsearch": query,
        "gsrnamespace": 6,        # namespace 'File:'
        "gsrlimit": limit,
        "prop": "imageinfo",
        "iiprop": "url|size|mime|extmetadata",
        "format": "json",
        "origin": "*"
    }

    resp = requests.get(url_api, params=params, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    if "query" not in data or "pages" not in data["query"]:
        return []

    images = []
    for page in data["query"]["pages"].values():
        title = page.get("title")  # "File:Something.jpg"
        imageinfo = page.get("imageinfo", [])
        if not imageinfo:
            continue
        info = imageinfo[0]
        url = info.get("url")
        width = info.get("width")
        height = info.get("height")
        mime = info.get("mime")
        extmeta = info.get("extmetadata", {})

        # Métadonnées de licence / auteur (si disponibles)
        license_short_name = extmeta.get("LicenseShortName", {}).get("value")
        license_url = extmeta.get("LicenseUrl", {}).get("value")
        artist = extmeta.get("Artist", {}).get("value")
        credit = extmeta.get("Credit", {}).get("value")

        images.append({
            "title": title,
            "url": url,
            "width": width,
            "height": height,
            "mime": mime,
            "license_short_name": license_short_name,
            "license_url": license_url,
            "artist": artist,
            "credit": credit,
            "source": "Wikimedia Commons"
        })

    return images

In [ ]:
all_sources = []
topics = [
    ("cat", 40),
    ("dog", 40),
    ("mountain landscape", 40),
]

for query, limit in topics:
    results = search_commons_images(query, HEADERS, limit=limit)
    all_sources.extend(results)

print(f"{len(all_sources)} images trouvées avant dédoublonnage")

In [10]:
# Facultatif : évite certains problèmes sur images tronquées
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Extensions acceptées comme images
VALID_IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp"}


def download_image(url, dest_folder, idx):
    response = requests.get(url, headers=HEADERS, timeout=10)
    response.raise_for_status()

    ext = os.path.splitext(url.split("?")[0])[1].lower()

    if not ext:
        ext = ".jpg"

    filename = f"img_{idx:03d}{ext}"
    filepath = os.path.join(dest_folder, filename)

    with open(filepath, "wb") as f:
        f.write(response.content)

    return filename, filepath, ext

def extract_exif(image_path):
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            with Image.open(image_path) as img:
                exif_data = img.getexif()

        if not exif_data:
            return {}

        exif = {}
        for tag_id, value in exif_data.items():
            tag = ExifTags.TAGS.get(tag_id, tag_id)
            exif[tag] = str(value)

        return exif

    except Exception:
        return {}

metadata_list = []

for idx, src in enumerate(all_sources, start=1):
    url = src.get("url")
    license_info = src.get("license", "unknown")
    source_site = src.get("source", "unknown")

    try:
        filename, filepath, ext = download_image(url, IMAGES_DIR, idx)
        time.sleep(0.75)

        # Ignorer les fichiers qui ne sont pas des images
        if ext not in VALID_IMAGE_EXTENSIONS:
            print(f"Image {idx} ignorée ({url}) : extension non supportée ({ext})")
            continue

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                with Image.open(filepath) as img:
                    width, height = img.size
                    img_format = img.format

        except UnidentifiedImageError:
            print(f"Erreur sur l’image {idx} ({url}) : fichier non reconnu comme image")
            continue

        except Image.DecompressionBombError:
            print(f"Erreur sur l’image {idx} ({url}) : image trop grande, ignorée")
            continue

        file_size_bytes = os.path.getsize(filepath)
        file_size_kb = round(file_size_bytes / 1024, 2)

        exif = extract_exif(filepath)

        metadata = {
            "filename": filename,
            "width": width,
            "height": height,
            "format": img_format,
            "file_size_kb": file_size_kb,
            "url": url,
            "license": license_info,
            "source": source_site,
            "exif": exif
        }
        metadata_list.append(metadata)

    except Exception as e:
        print(f"Erreur sur l’image {idx} ({url}): {e}")

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata_list, f, ensure_ascii=False, indent=2)

print(f"{len(metadata_list)} images traitées et sauvegardées dans {METADATA_PATH}")

Image 12 ignorée (https://upload.wikimedia.org/wikipedia/commons/a/ac/Cat_lapping_water_off_ground_in_slow_motion.gk.webm) : extension non supportée (.webm)
Image 60 ignorée (https://upload.wikimedia.org/wikipedia/commons/e/e6/Dog_penis.ogv) : extension non supportée (.ogv)
Erreur sur l’image 73 (https://upload.wikimedia.org/wikipedia/commons/8/82/P_S_Kr%C3%B8yer_1899_-_Sommeraften_ved_Skagens_strand._Kunstneren_og_hans_hustru.jpg) : image trop grande, ignorée
117 images traitées et sauvegardées dans .\data\images_metadata.json
